# 🧠 NovaMind Training Notebook — Build Your Own LLM

**Train a custom Large Language Model from scratch using pure PyTorch.**

This notebook walks you through the entire process:
1. Install dependencies
2. Set up the project
3. Collect training data
4. Train a BPE tokenizer
5. Create the dataset
6. Build and train the NovaMind model
7. Evaluate and generate text
8. Save to Google Drive

**Runtime**: Select GPU → Runtime → Change runtime type → T4 GPU

---
*Built by Purushottam — from Chennai to the stars* 🌟

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q torch numpy tqdm matplotlib requests ipywidgets

## 💾 Step 2: Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Create project directory on Drive for saving results
import os

DRIVE_DIR = "/content/drive/MyDrive/NovaMind"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive directory: {DRIVE_DIR}")

**Option A**: Clone from GitHub (Highly Recommended):
```python
!git clone https://github.com/PurushottamGH/NOVA.git /content/novamind
```


In [ ]:
import os

PROJECT_DIR = "/content/novamind"

if os.path.exists(PROJECT_DIR):
    print(f"Project already exists at {PROJECT_DIR}")
else:
    # Try cloning automatically
    !git clone https://github.com/PurushottamGH/NOVA.git {PROJECT_DIR}

    if not os.path.exists(PROJECT_DIR):
        # Try from Drive second
        drive_project = "/content/drive/MyDrive/NOVA"
        if os.path.exists(drive_project):
            !cp -r {drive_project} {PROJECT_DIR}
            print("Copied from Drive")
        else:
            print("Please upload the novamind folder to /content/novamind")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

## ⚙️ Step 4: Configuration

NovaMindConfig holds ALL hyperparameters. Here's what each one controls:

In [ ]:
import sys

sys.path.insert(0, "/content/novamind")

from model.config import NovaMindConfig

config = NovaMindConfig(
    # === Model Architecture ===
    vocab_size=8000,  # BPE vocabulary size (8K is good for small models)
    embed_dim=256,  # Hidden state dimensionality (256 is lightweight)
    num_heads=8,  # Attention heads (each head_dim = 256/8 = 32)
    num_layers=6,  # Transformer blocks stacked (6 is a good start)
    context_length=512,  # Max tokens the model can "see" at once
    feedforward_dim=1024,  # FFN inner dimension (4x embed_dim)
    dropout=0.1,  # Regularization strength
    weight_tying=True,  # Share embedding/output weights (saves params)
    # === Training ===
    learning_rate=3e-4,  # Peak learning rate (3e-4 is the "magic number")
    weight_decay=0.01,  # L2 regularization
    warmup_steps=100,  # Linear LR warmup steps
    max_steps=5000,  # Total training steps (increase for better quality)
    batch_size=16,  # Sequences per batch (reduce if OOM)
    grad_clip=1.0,  # Max gradient norm
    save_every=500,  # Checkpoint frequency
    eval_every=100,  # Evaluation frequency
    device="auto",  # Auto-detect: cuda > mps > cpu
)

print(config)

## 📥 Step 5: Collect Training Data

Downloads free text from:
- 📚 Project Gutenberg (classic books)
- 📖 Wikipedia (encyclopedia articles)
- 📄 arXiv (AI and astronomy paper abstracts)

In [ ]:
from data.collector import collect_all

total_chars = collect_all("personal_data")
print(f"\nTotal characters collected: {total_chars:,} ({total_chars / 1e6:.1f}M)")

In [ ]:
# Clean the collected data
from data.cleaner import clean_directory

clean_directory("personal_data", "personal_data")

## 🔤 Step 6: Train BPE Tokenizer

Byte Pair Encoding learns subword tokens from the training data.

In [ ]:
from data.dataloader import get_text_files
from tokenizer.tokenizer import NovaMindTokenizer

# Get all text files
text_files = get_text_files("personal_data")

# Train tokenizer
tokenizer = NovaMindTokenizer()
tokenizer.train(text_files, vocab_size=config.vocab_size)

# Save tokenizer
tokenizer.save("tokenizer_data")

# Update config with actual vocab size
config.vocab_size = tokenizer.vocab_size
print(f"Vocab size: {tokenizer.vocab_size}")

# Test encoding/decoding
test_text = "Hello, this is NovaMind speaking!"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"Original: {test_text}")
print(f"Encoded:  {encoded[:20]}... ({len(encoded)} tokens)")
print(f"Decoded:  {decoded}")

## 📊 Step 7: Create Dataset

Creates overlapping chunks using a sliding window for next-token prediction.

In [ ]:
from data.dataloader import create_dataloaders

train_loader, val_loader = create_dataloaders(
    text_files=text_files,
    tokenizer=tokenizer,
    config=config,
    train_split=0.9,
)

# Show a sample
sample_input, sample_target = next(iter(train_loader))
print(f"Input shape: {sample_input.shape}")
print(f"Target shape: {sample_target.shape}")
print(f"Sample input tokens: {sample_input[0, :10].tolist()}")
print(f"Sample target tokens: {sample_target[0, :10].tolist()}")

## 🧠 Step 8: Create NovaMind Model

Instantiate the full decoder-only transformer.

In [ ]:
from model.architecture import NovaMind
from model.utils import estimate_flops, model_summary

model = NovaMind(config)

# Print model summary
model_summary(model)

# Estimate FLOPs
estimate_flops(config)

# Parameter count
params = model.count_parameters()
print(f"\nTotal: {params['total_million']}M parameters")
print(f"Trainable: {params['trainable_million']}M parameters")

## 🏋️ Step 9: Train!

The trainer handles:
- Forward/backward passes
- Gradient clipping
- LR scheduling (cosine with warmup)
- Periodic evaluation and sample generation
- Checkpoint saving

⏱️ **Expected time**: ~10-30 minutes on T4 GPU for 5000 steps

In [ ]:
from training.trainer import Trainer

trainer = Trainer(
    model=model,
    config=config,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    checkpoint_dir="weights",
    log_dir="logs",
    resume=False,  # Set True to resume from checkpoint
)

trainer.train()

## 📉 Step 10: Loss Curve

In [ ]:
import os

from IPython.display import Image, display

loss_curve_path = "logs/loss_curve.png"
if os.path.exists(loss_curve_path):
    display(Image(filename=loss_curve_path))
else:
    print("Loss curve not found. Training may still be in progress.")
    # Generate it manually
    trainer.plot_losses()

## 📝 Step 11: Generate Text

Test the trained model with different prompts.

In [ ]:
from inference.generate import generate_text

prompts = [
    "The universe is",
    "Artificial intelligence will",
    "Stars are born when",
    "In the future",
    "The most important discovery",
]

print("=" * 60)
print("  NovaMind Text Generation")
print("=" * 60)

for prompt in prompts:
    generated = generate_text(
        model,
        tokenizer,
        prompt,
        max_new_tokens=100,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
    )
    print(f'\nPrompt: "{prompt}"')
    print(f'Output: "{generated[:300]}"')
    print("-" * 60)

## 💾 Step 12: Save to Google Drive

In [ ]:
import shutil

# Save model
model.save(f"{DRIVE_DIR}/final_model")

# Save tokenizer
tokenizer.save(f"{DRIVE_DIR}/tokenizer")

# Copy logs
if os.path.exists("logs"):
    shutil.copytree("logs", f"{DRIVE_DIR}/logs", dirs_exist_ok=True)

print(f"\n✅ Everything saved to Google Drive: {DRIVE_DIR}")
print("Contents:")
!ls -la {DRIVE_DIR}/

## 🚀 Step 12.5: Push Weights to GitHub
Automatically pushes the trained model to your GitHub repository.

In [ ]:
!git config --global user.email "purushottamr1629@gmail.com"
!git config --global user.name "PurushottamGH"
!git add -f /content/novamind/weights/
!git commit -m "Add trained weights from Colab"
!git push origin main

## 🔌 Step 13: FastAPI Integration

Here's exactly how to load NovaMind into your Nova FastAPI backend.

In [ ]:
# === FastAPI Integration Example ===
# Copy this into your Nova brain.py

integration_code = """
# brain.py — Nova's AI Brain
from inference.chat import NovaChatEngine

# Initialize the engine (do this once at startup)
engine = NovaChatEngine(
    model_path="weights/final_model",
    tokenizer_path="weights/tokenizer",
    device="cpu",  # Use "cuda" if GPU available
)

# In your FastAPI route:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List, Dict

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    history: List[Dict[str, str]] = []

@app.post("/chat")
async def chat(request: ChatRequest):
    response = engine.chat(request.message, request.history)
    return {"response": response}

# Streaming endpoint
from fastapi.responses import StreamingResponse

@app.get("/stream")
async def stream_chat(message: str):
    async def generate():
        for token in engine.stream_chat(message):
            yield f"data: {token}\\n\\n"
    return StreamingResponse(generate(), media_type="text/event-stream")
"""

print(integration_code)
print("\n" + "=" * 60)
print("Copy the above code into your Nova FastAPI backend!")
print("This replaces the Groq API call completely.")
print("=" * 60)

## ✅ Done!

You've just built and trained your own Large Language Model from scratch.

**Next steps:**
- Add more training data for better quality
- Increase `max_steps` for longer training
- Scale up `embed_dim` and `num_layers` for a larger model
- Fine-tune on conversation data for better chat ability

**Built with 💜 by Purushottam**

## 🔥 Nova Focused Retraining

Advanced SFT on specialized coding, math, and professional datasets.

In [ ]:
from model.config import NovaMindConfig

config = NovaMindConfig(
    vocab_size=8000,
    embed_dim=512,  # Bigger than before (was 256)
    num_heads=8,
    num_layers=8,  # More layers (was 6)
    context_length=512,
    feedforward_dim=2048,
    dropout=0.1,
    batch_size=8,
    accumulation_steps=8,
    max_steps=20000,  # Much more training
    warmup_steps=500,
    learning_rate=3e-4,
    gradient_checkpointing=True,
)

print("Config updated for High-Performance Focused training.")

In [ ]:
from data.dataloader import get_text_files
from data.nova_focused_collector import collect_nova_focused

# Collect the high-quality focused dataset
collect_nova_focused()
text_files = get_text_files("personal_data")
print(f"Training files: {len(text_files)}")

In [ ]:
# After base training, run SFT on the chat-formatted files
sft_files = [
    f
    for f in text_files
    if any(
        name in f
        for name in ["stackoverflow", "math_problems", "nova_sft_core", "alpaca", "openassistant"]
    )
]
print(f"SFT files identified: {len(sft_files)}")

# Then run training/sft_train.py on these files
print("Ready to initiate SFT refinement.")